In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

#dataset na het opschonen van missing values inlezen
df = pd.read_csv('Leesmateriaal_DU.csv')
df = df.reset_index(drop=True)
df = df.convert_dtypes()
df.dtypes

Titel                                string
Auteur                               string
Korte omschrijving                   string
Type materiaal                       string
Niveau                               string
genre                                string
onderwerp                            string
trefwoorden                          string
taal                                 string
aantal_paginas                        Int64
geschatte_leestijd_min                Int64
bron_url                             string
doelgroep                            string
gegevensstatus                       string
leescomplexiteit_score              Float64
interessecluster                     string
uitgever                             string
publicatiejaar                        Int64
beschikbaar_formaat                  string
cataloguscode                        string
leescomplexiteit_score_ontbreekt    boolean
dtype: object

In [2]:
def maak_zoektekst(df):
    kolommen = [
        "Korte omschrijving", "Type materiaal", "Niveau", "genre",
        "onderwerp", "trefwoorden", "taal", "doelgroep"
    ]

    zoektekst = df[kolommen].apply(
        lambda rij: " ".join(
            rij.dropna().astype(str).str.strip()
        ),
        axis=1
    )

    return zoektekst


df["tekst_gecombineerd"] = maak_zoektekst(df)

df["tekst_gecombineerd"].iloc[[0]]

0    Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. Boek 3F Fictie WOII WOII; onderduik Nederlands MBO
Name: tekst_gecombineerd, dtype: str

In [3]:
import re
import spacy

nlp = spacy.load("nl_core_news_lg")

In [4]:
df["tekst_lowercase"] = (
    df["tekst_gecombineerd"]
    .str.lower()
    .str.strip()
)

df["tekst_lowercase"].iloc[[0]]

0    dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam. haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. boek 3f fictie woii woii; onderduik nederlands mbo
Name: tekst_lowercase, dtype: str

In [5]:
def tekst_opschonen(tekst):
    # Verwijder losse cijfers, maar behoud bijvoorbeeld 2f en 3f
    tekst = re.sub(r"(?<![A-Za-zÀ-ÿ])\d+(?![A-Za-zÀ-ÿ])", " ", tekst)

    # Verwijder leestekens, maar behoud letters, cijfers en spaties
    tekst = re.sub(r"[^A-Za-zÀ-ÿ0-9\s]", " ", tekst)

    # Maak meerdere spaties één spatie
    tekst = re.sub(r"\s+", " ", tekst)

    return tekst.strip()

df["tekst_schoon"] = df["tekst_lowercase"].apply(tekst_opschonen)

df["tekst_schoon"].iloc[[0]]

0    dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam haar persoonlijke observaties geven een indringend beeld van angst hoop groei en het dagelijks leven onder vervolging boek 3f fictie woii woii onderduik nederlands mbo
Name: tekst_schoon, dtype: str

In [6]:
df["tokens"] = df["tekst_schoon"].apply(
    lambda tekst: [token.text for token in nlp(tekst)]
)

df["tokens"].iloc[[0]]

0    [dagboek, van, een, joods, meisje, dat, tijdens, de, tweede, wereldoorlog, onderduikt, in, amsterdam, haar, persoonlijke, observaties, geven, een, indringend, beeld, van, angst, hoop, groei, en, het, dagelijks, leven, onder, vervolging, boek, 3f, fictie, woii, woii, onderduik, nederlands, mbo]
Name: tokens, dtype: object

In [7]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stopwoorden = set(stopwords.words("dutch"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\addov\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
def verwijder_stopwoorden(tekst):
    tokens = tekst.split()

    return [
        token
        for token in tokens
        if token not in stopwoorden
    ]

df["tokens_zonder_stopwoorden"] = df["tekst_schoon"].apply(
    verwijder_stopwoorden
)

df["tokens_zonder_stopwoorden"].iloc[[0]]

0    [dagboek, joods, meisje, tijdens, tweede, wereldoorlog, onderduikt, amsterdam, persoonlijke, observaties, geven, indringend, beeld, angst, hoop, groei, dagelijks, leven, vervolging, boek, 3f, fictie, woii, woii, onderduik, nederlands, mbo]
Name: tokens_zonder_stopwoorden, dtype: object

In [9]:
def lemmatiseer(tekst):
    doc = nlp(tekst)

    resultaat = []

    for token in doc:
        is_niveau = re.fullmatch(r"\d+[a-zA-Z]", token.text)

        if token.text.lower() in stopwoorden:
            continue

        if token.is_alpha:
            resultaat.append(token.lemma_)
        elif is_niveau:
            resultaat.append(token.text.upper())

    return resultaat

df["tokens_basisvorm"] = df["tekst_schoon"].apply(lemmatiseer)

df["tokens_basisvorm"].iloc[[0]]

0    [dagboek, joods, meisje, tijdens, twee, wereldoorlog, onderduiken, Amsterdam, persoonlijk, observatie, geven, indringen, beeld, angst, hopen, groei, dagelijks, leven, vervolging, boek, 3F, fictie, woii, woii, onderduiken, Nederlands, mbo]
Name: tokens_basisvorm, dtype: object

In [10]:
df["tekst_nlp"] = df["tokens_basisvorm"].apply(
    lambda tokens: " ".join(tokens)
)

df["tekst_nlp"].iloc[0:3]

0    dagboek joods meisje tijdens twee wereldoorlog onderduiken Amsterdam persoonlijk observatie geven indringen beeld angst hopen groei dagelijks leven vervolging boek 3F fictie woii woii onderduiken Nederlands mbo
1                                    spannend thriller waarin vrouw confronteren stalking bedreiging terwijl spanning oplopen zetten zichzelf gezin beschermen boek 3F fictie spanning spanning stalking Nederlands mbo
2                                    toegankelijk verhaal liefde relatie zoektocht passen partner plattelandsleven vormen decor herkenbaar menselijk ervaring keuze boek 2F fictie liefde liefde relatie Nederlands mbo
Name: tekst_nlp, dtype: str

In [11]:
df.iloc[0:3]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,leescomplexiteit_score_ontbreekt,tekst_gecombineerd,tekst_lowercase,tekst_schoon,tokens,tokens_zonder_stopwoorden,tokens_basisvorm,tekst_nlp
0,Het dagboek van Anne Frank,Anne Frank,"Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging.",Boek,3F,Fictie,WOII,WOII; onderduik,Nederlands,265,556,<NA>,MBO,Beschikbaar,45.4,Verhalen,Educatief Media,2025,Pdf,CAT-00001,True,"Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. Boek 3F Fictie WOII WOII; onderduik Nederlands MBO","dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam. haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. boek 3f fictie woii woii; onderduik nederlands mbo",dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam haar persoonlijke observaties geven een indringend beeld van angst hoop groei en het dagelijks leven onder vervolging boek 3f fictie woii woii onderduik nederlands mbo,"[dagboek, van, een, joods, meisje, dat, tijdens, de, tweede, wereldoorlog, onderduikt, in, amsterdam, haar, persoonlijke, observaties, geven, een, indringend, beeld, van, angst, hoop, groei, en, het, dagelijks, leven, onder, vervolging, boek, 3f, fictie, woii, woii, onderduik, nederlands, mbo]","[dagboek, joods, meisje, tijdens, tweede, wereldoorlog, onderduikt, amsterdam, persoonlijke, observaties, geven, indringend, beeld, angst, hoop, groei, dagelijks, leven, vervolging, boek, 3f, fictie, woii, woii, onderduik, nederlands, mbo]","[dagboek, joods, meisje, tijdens, twee, wereldoorlog, onderduiken, Amsterdam, persoonlijk, observatie, geven, indringen, beeld, angst, hopen, groei, dagelijks, leven, vervolging, boek, 3F, fictie, woii, woii, onderduiken, Nederlands, mbo]",dagboek joods meisje tijdens twee wereldoorlog onderduiken Amsterdam persoonlijk observatie geven indringen beeld angst hopen groei dagelijks leven vervolging boek 3F fictie woii woii onderduiken Nederlands mbo
1,Blauw Water,S. van der Vlugt,"Spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging. Terwijl de spanning oploopt, moet zij alles op alles zetten om zichzelf en haar gezin te beschermen.",Boek,3F,Fictie,spanning,spanning; stalking,Nederlands,171,184,<NA>,MBO / NT2,Beschikbaar,48.8,Verhalen,Leeswereld,2005,Online,CAT-00002,False,"Spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging. Terwijl de spanning oploopt, moet zij alles op alles zetten om zichzelf en haar gezin te beschermen. Boek 3F Fictie spanning spanning; stalking Nederlands MBO / NT2","spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging. terwijl de spanning oploopt, moet zij alles op alles zetten om zichzelf en haar gezin te beschermen. boek 3f fictie spanning spanning; stalking nederlands mbo / nt2",spannende thriller waarin een vrouw wordt geconfronteerd met stalking en bedreiging terwijl de spanning oploopt moet zij alles op alles zetten om zichzelf en haar gezin te beschermen boek 3f fictie spanning spanning stalking nederlands mbo nt2,"[spannende, thriller, waarin, een, vrouw, wordt, geconfronteerd, met, stalking, en, bedreiging, terwijl, de, spanning, oploopt, moet, zij, alles, op, alles, zetten, om, zichzelf, en, haar, gezin, te, beschermen, boek, 3f, fictie, spanning, spanning, stalking, nederlands, mbo, nt2]","[spannende, thriller, waarin, vrouw, geconfronteerd,

In [12]:
print(type(df))
print(len(df))
print(df.shape)

<class 'pandas.DataFrame'>
277
(277, 28)


In [13]:
print("Type df:", type(df))
print("Aantal rijen df:", len(df))
print("Aantal teksten:", len(df["tekst_nlp"]))
print("Lege teksten:", df["tekst_nlp"].isna().sum())

Type df: <class 'pandas.DataFrame'>
Aantal rijen df: 277
Aantal teksten: 277
Lege teksten: 0


In [14]:
print("Aantal rijen df:", len(df))
print("Aantal teksten:", len(df["tekst_nlp"]))
print("Lege teksten:", df["tekst_nlp"].isna().sum())
print("Index df:", df.index[:5])

Aantal rijen df: 277
Aantal teksten: 277
Lege teksten: 0
Index df: RangeIndex(start=0, stop=5, step=1)


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b"
)

tfidf_matrix = vectorizer.fit_transform(df["tekst_nlp"])

tfidf_matrix.shape

(277, 753)

In [16]:
woorden = vectorizer.get_feature_names_out()

woorden[:20]

array(['2F', '3F', 'Amsterdam', 'Anne', 'Bedrog', 'Biodiversiteit',
       'Dichtbundel', 'Duits', 'Eerstejaars', 'Engeland', 'Europees',
       'Fictieboek', 'Holmes', 'Kansenongelijkheid', 'Maatschappij',
       'Migratie', 'Nederland', 'Nederlands', 'Rika', 'Sport'],
      dtype=object)

In [17]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=woorden,
    index=df.index
)

tfidf_df.head()

,2F,3F,Amsterdam,Anne,Bedrog,Biodiversiteit,Dichtbundel,Duits,Eerstejaars,Engeland,Europees,Fictieboek,Holmes,Kansenongelijkheid,Maatschappij,Migratie,Nederland,Nederlands,Rika,Sport,Verlies,Wereldoorlog,aandacht,aangrijpen,aanleiding,aantrekkingkracht,aanwijzing,academisch,acceptatie,achtergrond,achtergrondinformatie,achterhalen,achtervolging,actualiteit,actueel,administratie,advocaat,af,afhankelijkheid,afscheid,afsluiten,afspelen,afspraak,afstaan,agent,ageverhaal,ai,alledaags,ambitie,analyseren,anders,angst,arbeidmarkt,arbeidsmarken,artikel,auteur,authenticiteit,avontuur,balans,band,bedreigen,bedreiging,bedrog,beeld,beeldvorming,begin,beginnen,begrijpelijk,behoefte,bekend,bekentenis,beknen,belangrijk,belichten,benauwen,bepalen,beproefd,beroemd,beschaving,beschermen,beschouwing,beschrijven,beslissing,bestaan,bestemming,betekenis,betekenisvol,betrekken,bewerking,bewijsvoering,beïnvloeen,bieden,bijzonder,binnen,biografie,blijken,blijven,blik,blogpost,blootleggen,bod,boek,botsen,brengen,brief,broer,bundel,bureaucratie,burger,burgerschap,buurman,centraal,coming,communicatie,competitie,complex,concentratiekamp,confronteren,consequentie,consumentenrecht,creativiteit,criminaliteit,cultuur,cybercrime,daarbij,daarvan,dader,dagboek,dagelijks,decor,deel,deelnemen,deepfakes,delen,democratie,denken,detail,detective,detectiveroman,detectiveverha,dichtbij,dichtbundel,dichterbij,dienstbaarheid,digitaal,digitalisering,dilemma,diner,discriminatie,discussie,diversiteit,doel,doorwerken,doorzetten,doorzettingsvermog,draad,droom,drugscriminaliteit,druk,duidelijk,durven,dystopisch,echt,echtpaar,economie,eenvoudig,eigen,eigenzinnig,eindelijk,elkaar,emotioneel,ervaren,ervaring,ervoor,essentieel,eten,extra,faillissement,familie,familiebanden,familiebedrijf,familiegehei,familiegeheime,familiegeschiedenis,familielid,familieverhaal,favoriet,feest,feit,fictie,fictieverhaal,filosofie,filosofisch,financieel,financiën,frank,fraude,fysiek,fysiotherapie,gaan,gang,gebaar,gebeuren,gebeurtenis,geborgenheid,gedrag,geheim,geheimen,geld,gelijk,geloven,geluk,gender,generatie,generatieverschil,geruchtmaken,geschiedenis,geschikt,gesprek,gevaar,gevaarlijk,gevangeniss,geven,gevoelen,gevolg,geweld,gewoon,gezin,gezinssituatie,gezondheid,goed,graphic,groei,groeien,groep,groepsdruk,grondrecht,groot,halen,hand,handelen,heden,heen,helder,helpen,hen,herinnering,herkenbaar,herkenning,herstel,historisch,hoofd,hoofdpersoon,hoog,hoogleraar,hoop,hopen,houvast,huisarts,huiselijk,humor,humoristisch,idee,identiteit,ieder,iedereen,illustratie,impact,individuen,indringen,indringend,informatief,informatieften,ingewikkeld,ingrijpen,inkijk,innovatie,inspiratie,inspirerend,integratie,interactief,internationaal,intrige,inventiviteit,invloed,inzicht,jeugdcriminaliteit,jeugdroman,jeugdthriller,jeugdzorg,jezelf,jong,jongen,joods,juridisch,kanker,kans,kennen,kennis,kennismaken,kern,keuze,keuzes,kijk,kijken,kind,klassiek,klein,klimaat,komen,kort,kosten,krant,krijgen,kruis,kwetsbaarhed,laaggeletterde,laat,landelijk,langzaam,laten,leerling,leesmateriaal,leiden,leren,leven,levensles,levensless,levenslessen,levensverhaal,levensverhal,levensvrag,levenswijsheid,lezer,licht,liefde,lijken,locatie,logica,lokaal,lopen,lot,loyaliteit,luchtmacht,maatschappelijk,macht,machteloosheid,machtverhouding,maken,man,manier,mannelijkheid,mbo,mediaoptreden,medium,mee,meerdere,meisje,melding,mens,menselijk,menselijkheid,mentaal,migratie,misdaad,missie,misverstand,moed,moeder,moederschap,moedig,moeilijk,moeten,moment,mooi,moordzaak,moraal,moreel,motief,muziek,mysterie,mysterieuz,nacht,nadenken,nalatenschap,natuur,nederlander,nederlanders,newadultroman,nieuw,nieuws,nieuwsgierigheid,nodig,nodigen,non,normaal,novel,novelbewerking,novelle,observatie,offer,ogenschijnlijk,omgaan,omgeving,omstandigheid,omzwerving,onbekend,ondanks,onderdrukking,onderduiken,onderduikperiode,onderwerp,onderwijs,onderzoek,onderzoeken,onderzoeker,ongelijkheid,onhandig,online,onschuldig,ontdekken,onthullen,onthulling,ontknoping,ontmoeten,ontmoeting,on

In [18]:
boek_1 = tfidf_df.iloc[0]

display(
    boek_1[boek_1 > 0]
    .sort_values(ascending=False)
)

woii            0.437407
onderduiken     0.407521
beeld           0.218703
dagboek         0.218703
wereldoorlog    0.218703
vervolging      0.218703
joods           0.218703
Amsterdam       0.218703
angst           0.203761
observatie      0.203761
geven           0.193159
dagelijks       0.178216
hopen           0.172535
groei           0.167614
tijdens         0.167614
indringen       0.167614
meisje          0.159391
twee            0.149722
leven           0.122786
persoonlijk     0.122786
3F              0.062398
fictie          0.050493
boek            0.047543
mbo             0.037657
Nederlands      0.036853
Name: 0, dtype: float64

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_matrix.shape

(277, 277)

In [26]:
vergelijkbare_boeken = similarity_matrix[0].argsort()[::-1][1:6]

In [23]:
resultaat = pd.DataFrame({
    "index": vergelijkbare_boeken,
    "similarity": similarity_matrix[0, vergelijkbare_boeken],
    "tekst": df.iloc[vergelijkbare_boeken]["tekst_nlp"].to_numpy()
})

display(resultaat)

,index,similarity,tekst
0,241,0.285738,novelle waarin onderduiken spanning centraal stilstaan moed spanning vormen rood draad boek 2F fictie onderduiken onderduiken spanning Nederlands mbo
1,162,0.150205,liefde werk dagelijks leven dichtbundel 3F poëzie Nederlands mbo
2,84,0.149303,historisch roman jong meisje tijdens twee Wereldoorlog geheim moed overlevingskracht bepalen lot boek 3F fictie Nederlands mbo
3,128,0.145738,indringen roman identiteit familie persoonlijk groei personage zoeken verbinding betekenis complex wereld boek 3F fictie roman Nederlands mbo
4,58,0.134329,romantisch verhaal liefde timing persoonlijk groei twee jong proberen gevoelen toekomstplanan elkaar verenigen boek 2F fictie roman liefde liefde groei Nederlands mbo
